# F1 Analytics — Notebook 01: EDA & Data Loading
**Race:** 2024 Japanese Grand Prix · Suzuka

---

## What this notebook covers
This is the first step in the CRISP-DM process:
- **Business Understanding** — what question are we answering?
- **Data Understanding** — what data do we have, what does it look like?
- **Exploratory Data Analysis (EDA)** — visualize before modeling

From your course: *"Always visualize before modeling. Summary statistics alone can be deceiving (Anscombe's Quartet)."*

---

## Business Understanding
**Question:** How did Piastri and Verstappen compare in the 2024 Japanese GP?
- Who was faster on pure pace?
- How did lap times evolve across the race?
- What do sector times tell us about driving style?

This directly extends your IB IA — instead of modeling one corner, we analyze the full race.

---
## Step 1 — Import Libraries

**What is an import?**  
Python on its own can do basic math and logic. But for data analysis, charts, and F1 data — those aren't built in. Libraries are pre-written code that other people built and published for free. `import` loads them into your session so you can use them.

**Why these specific libraries and not others?**

| Library | Why this one |
|---|---|
| `fastf1` | The only Python library that gives you official F1 timing data. No alternative for this. |
| `pandas` | Industry standard for tabular data in Python. Every DA job uses it. Alternative: raw Python lists — but that's 10x more code for the same result. |
| `matplotlib` | The foundation of Python plotting. Every other plotting library is built on top of it. You can't avoid it. |
| `seaborn` | Built on matplotlib. Needs fewer lines for common chart types. We use it when our data is already in a DataFrame. |
| `numpy` | Math on arrays (lists of numbers). pandas uses it under the hood. We use it here for `np.arange()` which generates evenly spaced numbers. |

**Why `import pandas as pd` and not just `import pandas`?**  
`pd` is just a shorter nickname. Without it you'd write `pandas.DataFrame()` every time instead of `pd.DataFrame()`. This is a universal convention — every data scientist writes `pd`, `plt`, `sns`, `np`. If you see someone else's code, they'll use the same abbreviations.

In [ ]:
%matplotlib inline
import fastf1
import fastf1.plotting
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np

print("All libraries imported successfully.")

**What is `%matplotlib inline`?**  
This is a Jupyter-specific command (called a "magic command" — starts with `%`). Without it, charts might open in a separate window or not appear at all. `inline` tells Jupyter to draw charts directly inside the notebook, below the cell. Always put this at the top when doing data visualization in Jupyter.

---
## Step 2 — Enable Cache

**What is a cache?**  
FastF1 fetches race data from the official F1 API over the internet. The first time, it downloads everything — this takes 2–5 minutes. The cache saves a local copy on your computer. Every time after that, it reads from the local file instead — instant.

**Why `'cache'` as the folder name?**  
This is just a relative path — it means "create a folder called `cache` in the same folder as this notebook." You can call it anything. `'cache'` is the convention.

**What happens if you skip this line?**  
It re-downloads the data every single time you run the notebook. Slow and unnecessary.

In [ ]:
fastf1.Cache.enable_cache('cache')

print("Cache enabled. Data will be saved to the 'cache' folder.")

---
## Step 3 — Load the Session

**What does `get_session()` do?**  
It tells FastF1 which race and session you want. Think of it as placing an order — you haven't received anything yet, you've just specified what you want.

**What does `.load()` do?**  
This is the actual fetch — it goes to the cache (or API if cache is empty) and downloads everything: lap times, sector times, tire data, telemetry, weather, race control messages. Without `.load()`, the session object is empty.

**Why do we print the session info afterwards?**  
To verify we got what we asked for. Always confirm before you start working — it takes 1 second and prevents 20 minutes of debugging the wrong dataset.

In [ ]:
session = fastf1.get_session(2024, 'Japanese Grand Prix', 'R')
session.load()

print(f"Session loaded: {session.event['EventName']} {session.event.year}")
print(f"Circuit: {session.event['Location']}")
print(f"Date: {session.event['EventDate'].date()}")

---
## Step 4 — Data Understanding: What Does the Dataset Look Like?

**What is `session.laps`?**  
FastF1 organizes the loaded data into different objects. `.laps` gives you a pandas DataFrame where each row = one lap driven by one driver. If there are 20 drivers and 53 laps each, you get roughly 1,060 rows.

**Why `.shape` first?**  
Before doing anything with data, you need to know how big it is. `.shape` returns `(rows, columns)` — two numbers that tell you the scale of what you're working with. If you expected 1,000 rows and get 10, something went wrong with the load.

**Why `.columns`?**  
You can't work with data you don't know exists. Listing columns first tells you what features are available — so you know what questions you can ask.

In [ ]:
laps = session.laps

print(f"Total rows (laps): {laps.shape[0]}")
print(f"Total columns (features): {laps.shape[1]}")
print()
print("Available columns:")
print(list(laps.columns))

**Why `.head(5)` and not printing the whole table?**  
A race has ~1,000 rows. Printing all of them would flood the screen and tell you nothing useful. The first 5 rows are enough to see the structure — column names, data types, what the values look like. This is a standard first move in any data project.

In [ ]:
laps.head(5)

---
## Step 5 — Data Quality Check

**Why check for missing values at all?**  
From your course: *"Dirty data leads to inaccurate analytics results."* If a lap has no recorded `LapTime` and you try to calculate an average, Python either crashes or gives you a wrong answer silently. You need to know what's missing before you do any math.

**What causes missing values in F1 data?**  
- Driver retired mid-lap (DNF) — that lap has no finish time
- Safety car laps — timing is unusual
- Formation lap — not a racing lap
- Pit stop laps — the in-lap and out-lap are recorded but not representative of race pace

**What does `.isnull().sum()` do?**  
- `.isnull()` returns True/False for every cell — True if the value is missing
- `.sum()` counts the Trues — in Python, True = 1 and False = 0, so summing gives you the count of missing values

In [ ]:
key_columns = ['Driver', 'LapTime', 'LapNumber', 'Compound', 'TyreLife', 'Sector1Time', 'Sector2Time', 'Sector3Time']

print("Missing values per key column:")
print(laps[key_columns].isnull().sum())
print()

missing_pct = laps['LapTime'].isnull().sum() / len(laps) * 100
print(f"LapTime missing: {missing_pct:.1f}% of all laps")

**Why `pick_accurate()` instead of just `dropna()` (drop all rows with missing values)?**  

This is an important decision. Two options:

| Option | What it does | Problem |
|---|---|---|
| `dropna()` | Removes any row where ANY column has a missing value | Too aggressive — removes rows that are fine except for one unimportant column |
| `pick_accurate()` | FastF1's built-in filter — removes laps flagged as unrepresentative of race pace | Targeted — only removes pit laps, safety car laps, formation lap |

`pick_accurate()` is smarter because it uses FastF1's own logic about which laps are race-representative. A pit stop lap might have a valid `LapTime` value — it's not missing — but it's 30 seconds slower than a racing lap and would skew our analysis. `dropna()` wouldn't catch that. `pick_accurate()` does.

In [ ]:
clean_laps = laps.pick_accurate()

print(f"Total laps: {len(laps)}")
print(f"Clean (accurate) laps: {len(clean_laps)}")
print(f"Removed: {len(laps) - len(clean_laps)} laps (pit laps, SC laps, formation lap)")

---
## Step 6 — Filter to Piastri and Verstappen

**Why filter to just two drivers?**  
We have 20 drivers in the data. Our business question is specifically about Piastri vs Verstappen — the two drivers from your IB IA. Filtering makes charts readable and analysis focused. Showing all 20 drivers in one chart would be unreadable noise.

**Why `.copy()` after filtering?**  
When you filter a pandas DataFrame, the result is a "view" — it's still connected to the original. If you then try to add a new column to it, pandas throws a warning: *"SettingWithCopyWarning"* — because it's not sure if you want to modify the original or the copy. Calling `.copy()` makes it a fully independent DataFrame and avoids that warning entirely.

**Why convert LapTime to seconds? Why not keep it as-is?**  
FastF1 stores `LapTime` as a `timedelta` — Python's format for durations (like `0:01:33.456`). That format is human-readable but you cannot do math on it directly. You can't calculate an average of timedeltas easily, you can't plot them on a chart, and you can't feed them into a machine learning model later. Converting to a plain number (93.456 seconds) solves all of that.

**What does `.dt.total_seconds()` mean?**  
- `.dt` = "this column contains date/time data, access its time methods"
- `.total_seconds()` = convert the duration to a single float number in seconds

In [ ]:
ver_laps = clean_laps.pick_drivers('VER').copy()
pia_laps = clean_laps.pick_drivers('PIA').copy()

print(f"Verstappen clean laps: {len(ver_laps)}")
print(f"Piastri clean laps: {len(pia_laps)}")
print()

# Create a new column with lap time as a plain number (seconds)
ver_laps['LapTimeSec'] = ver_laps['LapTime'].dt.total_seconds()
pia_laps['LapTimeSec'] = pia_laps['LapTime'].dt.total_seconds()

print("Sample — Verstappen lap times:")
print(ver_laps[['LapNumber', 'LapTimeSec', 'Compound', 'TyreLife']].head(8))

---
## Step 7 — Descriptive Statistics

**What does `.describe()` give you?**  
Eight numbers that summarize the whole column:

| Output | What it means |
|---|---|
| `count` | How many non-missing values |
| `mean` | Average |
| `std` | Standard deviation — how spread out the values are |
| `min` | Fastest lap |
| `25%` | 25th percentile — 25% of laps were faster than this |
| `50%` | Median — the middle value |
| `75%` | 75th percentile — 75% of laps were faster than this |
| `max` | Slowest lap |

**Why do we calculate the gap using mean and not median here?**  
For a quick first look, mean is fine. We'll switch to median in the charts because median is more robust — one unusually slow lap (e.g. a slow out-lap after a pit stop that slipped through the filter) would drag the mean down but wouldn't affect the median.

**What does `{gap:+.3f}` mean in the print statement?**  
- `+` = always show the sign (+ or -), so you can immediately see who is faster
- `.3f` = 3 decimal places, `f` = floating point number  
Positive gap = VER is slower. Negative gap = VER is faster.

In [ ]:
print("=== Verstappen Lap Time Stats (seconds) ===")
print(ver_laps['LapTimeSec'].describe().round(3))
print()
print("=== Piastri Lap Time Stats (seconds) ===")
print(pia_laps['LapTimeSec'].describe().round(3))
print()

gap = ver_laps['LapTimeSec'].mean() - pia_laps['LapTimeSec'].mean()
print(f"Mean lap time gap (VER - PIA): {gap:+.3f} seconds")
print("Positive = Verstappen slower on average. Negative = Verstappen faster.")

---
## How to Read Plotting Code

> **Nobody memorizes plotting syntax. Not even senior data scientists.**  
> The real skill is knowing what chart you want, then finding or adapting code for it.

### The mental model — every matplotlib chart follows this exact 6-step pattern:

```python
fig, ax = plt.subplots(figsize=(14, 6))   # 1. Create the canvas + drawing area
ax.plot(x, y, ...)                         # 2. Draw something on it
ax.set_xlabel('...')                       # 3. Label x-axis
ax.set_ylabel('...')                       # 4. Label y-axis
ax.set_title('...')                        # 5. Add a title
plt.show()                                 # 6. Render it
```

- `fig` = the picture frame (the whole image)
- `ax` = the whiteboard inside the frame (where drawing happens)
- Everything starting with `ax.` = "do something on the whiteboard"
- `figsize=(14, 6)` = 14 inches wide, 6 inches tall

### Breaking down the most complex-looking line:

```python
ax.plot(ver_laps['LapNumber'], ver_laps['LapTimeSec'],
        color='#0600EF', marker='o', markersize=3, linewidth=2, label='Verstappen')
```

| Part | What it does | How you'd find this |
|---|---|---|
| `ax.plot(x, y)` | Draw a line. First arg = x-axis data, second = y-axis data | Google: "matplotlib line chart" |
| `color='#0600EF'` | Blue — Verstappen's Red Bull colour (hex code) | Google: "hex color picker" |
| `marker='o'` | Draw a dot at each data point | Google: "matplotlib markers list" |
| `markersize=3` | Size of the dot | Trial and error |
| `linewidth=2` | Thickness of the line | Trial and error |
| `label=` | What shows up in the legend | Google: "matplotlib legend" |

### matplotlib vs seaborn — why two libraries?

| | matplotlib | seaborn |
|---|---|---|
| Best for | Line charts, custom scatter plots | Box plots, grouped charts, anything where data is in a DataFrame |
| Code amount | More lines, more control | Fewer lines, smarter defaults |
| Decision rule | Use when you're plotting x vs y manually | Use when seaborn can read your DataFrame column directly |

### Your real workflow when building a new chart:
1. Decide what you want → *"two lines on one chart, different colours"*
2. Google → *"matplotlib two lines different colors"*
3. Copy the simplest example you find
4. Swap their variable names for yours
5. Adjust `color=`, `linewidth=`, `figsize=` until it looks right

After doing this 3–4 times you start remembering it naturally.

ver_plot = ver_laps[['LapTimeSec']].copy()
ver_plot['Driver'] = 'Verstappen'

pia_plot = pia_laps[['LapTimeSec']].copy()
pia_plot['Driver'] = 'Piastri'

combined = pd.concat([ver_plot, pia_plot], ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 6))

# hue='Driver' + legend=False is the correct seaborn syntax for colour-by-group without a legend
# (the legend is redundant here since the x-axis already labels the drivers)
sns.boxplot(
    data=combined,
    x='Driver',
    y='LapTimeSec',
    hue='Driver',
    palette={'Verstappen': '#0600EF', 'Piastri': '#FF8000'},
    legend=False,
    ax=ax
)

ax.set_xlabel('Driver', fontsize=12)
ax.set_ylabel('Lap Time (seconds)', fontsize=12)
ax.set_title('2024 Japanese GP — Lap Time Distribution', fontsize=14, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/02_lap_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Draw a line for each driver
# x = lap number (horizontal axis), y = lap time in seconds (vertical axis)
ax.plot(ver_laps['LapNumber'], ver_laps['LapTimeSec'],
        color='#0600EF',  # Red Bull blue
        marker='o',       # dot at each lap
        markersize=3,     # small — 53 dots would clutter at large size
        linewidth=2,
        label='Verstappen (VER)')

ax.plot(pia_laps['LapNumber'], pia_laps['LapTimeSec'],
        color='#FF8000',  # McLaren orange
        marker='o',
        markersize=3,
        linewidth=2,
        label='Piastri (PIA)')

ax.set_xlabel('Lap Number', fontsize=12)
ax.set_ylabel('Lap Time (seconds)', fontsize=12)
ax.set_title('2024 Japanese GP — Lap Time Comparison: Verstappen vs Piastri', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)    # draws the legend — uses the label= values from ax.plot above
ax.grid(True, alpha=0.3)  # alpha=0.3 = 30% opacity, so grid is visible but not distracting

plt.tight_layout()  # prevents axis labels from being cut off at the edges
plt.savefig('outputs/01_lap_time_trend.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 9 — Chart 2: Lap Time Distribution (Box Plot)

**Why a different chart — don't we already have the lap times from the line chart?**  
The line chart shows *when* each lap happened. The box plot answers a different question: *where do most laps cluster?* It ignores time order and focuses on the statistical spread.

**Why not a histogram instead?**  
A histogram would also show distribution. But it works best for one group at a time. The box plot is compact enough to compare two groups side by side clearly — one box per driver.

**Box plot anatomy:**
```
    |           ← whisker top (max, excluding outliers)
  ┌─┴─┐
  │   │        ← box top = 75th percentile (75% of laps were slower than this)
  │───│        ← middle line = median (50% above, 50% below)
  │   │        ← box bottom = 25th percentile
  └─┬─┘
    |           ← whisker bottom (min, excluding outliers)
    o           ← outlier dot (very slow lap that didn't get filtered)
```

**Why do we need to combine the two DataFrames into one before plotting?**  
Seaborn's `boxplot` function expects one DataFrame with a column that identifies the group (`'Driver'`). It reads that column and automatically draws one box per unique value. If you give it two separate DataFrames, it doesn't know how to split them. So we add a `'Driver'` label column to each, then stack them with `pd.concat`.

**What does `pd.concat` do?**  
It stacks DataFrames on top of each other (vertically). Like copy-pasting two spreadsheet tables one below the other. `ignore_index=True` resets the row numbers so they go 0, 1, 2... instead of repeating.

In [ ]:
# Add a 'Driver' label column to each, then stack into one DataFrame
ver_plot = ver_laps[['LapTimeSec']].copy()
ver_plot['Driver'] = 'Verstappen'

pia_plot = pia_laps[['LapTimeSec']].copy()
pia_plot['Driver'] = 'Piastri'

combined = pd.concat([ver_plot, pia_plot], ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 6))

# seaborn reads x='Driver' and draws one box per unique driver name automatically
# palette = dictionary: driver name → colour
sns.boxplot(
    data=combined,
    x='Driver',
    y='LapTimeSec',
    palette={'Verstappen': '#0600EF', 'Piastri': '#FF8000'},
    ax=ax
)

ax.set_xlabel('Driver', fontsize=12)
ax.set_ylabel('Lap Time (seconds)', fontsize=12)
ax.set_title('2024 Japanese GP — Lap Time Distribution', fontsize=14, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)  # axis='y' = only horizontal grid lines

plt.tight_layout()
plt.savefig('outputs/02_lap_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 10 — Chart 3: Sector Times Comparison (Bar Chart)

**Why sector times?**  
A lap time is the sum of all three sector times. Comparing only total lap time tells you who is faster overall, but not *where* on track. Sectors tell you if one driver is better in high-speed corners (Sector 1) vs slow corners (Sector 2).

Suzuka's sectors:
- **Sector 1** — Turn 1 + Esses — the corner from your IB IA is here
- **Sector 2** — Degner, Hairpin, Spoon — heavy braking zones
- **Sector 3** — 130R + Casio Triangle — mixed

**Why median sector time and not mean?**  
One unusually slow sector (e.g. a moment where a driver encountered traffic) would pull the mean down and make it look like that driver is generally slower in that sector. The median is the middle value — half above, half below. It's resistant to those occasional bad sectors and gives a better picture of typical pace.

**Why a bar chart here and not a line chart?**  
Sectors are categories (1, 2, 3) — they don't have a natural order the way lap numbers do. There's no progression from Sector 1 to Sector 2. A bar chart is the right tool when comparing discrete categories.

**What is `np.arange(len(sectors))` and why do we need it?**  
Bar charts in matplotlib need an x position (a number) for each bar. `np.arange(3)` gives `[0, 1, 2]` — three positions. We then shift one driver's bars left by `width/2` and the other right by `width/2` so they sit side by side instead of on top of each other. Seaborn would handle this automatically, but matplotlib gives us more control over exact positioning.

In [ ]:
# Convert sector times from timedelta to seconds — same reason as LapTime
for driver_laps in [ver_laps, pia_laps]:
    for sector in ['Sector1Time', 'Sector2Time', 'Sector3Time']:
        col = sector.replace('Time', 'Sec')  # e.g. 'Sector1Time' → 'Sector1Sec'
        driver_laps[col] = driver_laps[sector].dt.total_seconds()

sectors = ['Sector1Sec', 'Sector2Sec', 'Sector3Sec']
sector_labels = ['Sector 1\n(Turn 1 + Esses)', 'Sector 2\n(Degner + Hairpin)', 'Sector 3\n(130R + Casio)']

# Median per sector per driver — more robust than mean
ver_medians = [ver_laps[s].median() for s in sectors]
pia_medians = [pia_laps[s].median() for s in sectors]

# Print raw numbers first — always sanity check before charting
print("Median sector times (seconds):")
print(f"{'Sector':<12} {'Verstappen':>12} {'Piastri':>10} {'Gap (V-P)':>12}")
print("-" * 50)
for label, v, p in zip(['Sector 1', 'Sector 2', 'Sector 3'], ver_medians, pia_medians):
    print(f"{label:<12} {v:>12.3f} {p:>10.3f} {v-p:>+12.3f}")

In [ ]:
# [0, 1, 2] — one position on the x-axis per sector
x = np.arange(len(sectors))
width = 0.35  # how wide each individual bar is

fig, ax = plt.subplots(figsize=(10, 6))

# Shift VER bars left of centre, PIA bars right of centre
bars1 = ax.bar(x - width/2, ver_medians, width, label='Verstappen', color='#0600EF', alpha=0.85)
bars2 = ax.bar(x + width/2, pia_medians, width, label='Piastri', color='#FF8000', alpha=0.85)

# Add the value as text on top of each bar
# bar.get_x() + bar.get_width()/2 = horizontal centre of the bar
# bar.get_height() = the top of the bar = the value
# + 0.05 = a tiny gap so the text doesn't touch the bar
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.2f}s', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.2f}s', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Suzuka Sector', fontsize=12)
ax.set_ylabel('Median Sector Time (seconds)', fontsize=12)
ax.set_title('2024 Japanese GP — Sector Time Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)               # tell matplotlib where to put tick marks
ax.set_xticklabels(sector_labels)  # tell matplotlib what text to show at each tick
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0, max(ver_medians + pia_medians) * 1.15)  # 15% headroom so text labels fit

plt.tight_layout()
plt.savefig('outputs/03_sector_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 11 — Chart 4: Pace by Tire Compound (Grouped Box Plot)

**Why look at tire compound separately?**  
Different tire compounds are different tools — Soft is fastest but wears quickly, Medium and Hard are slower but last longer. If you compare lap times without accounting for compound, you might conclude one driver is faster when really they just ran Soft tires longer. Grouping by compound makes the comparison fair.

**What is `hue=` in seaborn?**  
Without `hue`, you'd get one box per compound — no driver split. `hue='Driver'` adds a second grouping dimension: for each compound, draw one box per driver side by side. It's seaborn's built-in equivalent of what we did manually with `x - width/2` in the bar chart above.

**Why `dropna(subset=['Compound'])`?**  
Some laps in the data don't have a compound recorded (usually the first lap before tire data is transmitted). If we leave those in, seaborn would create a group called `NaN` ("not a number") which is meaningless. `dropna(subset=['Compound'])` removes only the rows where `Compound` is missing — it doesn't touch rows where other columns are missing.

In [ ]:
ver_compound = ver_laps[['LapTimeSec', 'Compound']].copy()
ver_compound['Driver'] = 'Verstappen'

pia_compound = pia_laps[['LapTimeSec', 'Compound']].copy()
pia_compound['Driver'] = 'Piastri'

compound_data = pd.concat([ver_compound, pia_compound], ignore_index=True)

# Remove rows where Compound is unknown — would create a meaningless 'NaN' group in the chart
compound_data = compound_data.dropna(subset=['Compound'])

print("Lap count per driver per compound:")
print(compound_data.groupby(['Driver', 'Compound']).size().unstack(fill_value=0))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=compound_data,
    x='Compound',      # one group per tire type on x-axis
    y='LapTimeSec',    # lap time on y-axis
    hue='Driver',      # split each compound group into one box per driver
    palette={'Verstappen': '#0600EF', 'Piastri': '#FF8000'},
    ax=ax
)

ax.set_xlabel('Tire Compound', fontsize=12)
ax.set_ylabel('Lap Time (seconds)', fontsize=12)
ax.set_title('2024 Japanese GP — Pace by Tire Compound', fontsize=14, fontweight='bold')
ax.legend(title='Driver', fontsize=11)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/04_pace_by_compound.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 12 — EDA Summary

From your course: *EDA = find patterns, spot anomalies, test assumptions.*

Run this cell last. Read the numbers, then go back and look at the charts again — the numbers and visuals should tell the same story.

In [ ]:
print("=" * 60)
print("EDA SUMMARY — 2024 Japanese GP")
print("=" * 60)
print()
print(f"Total clean laps analyzed: {len(clean_laps)}")
print(f"Verstappen clean laps: {len(ver_laps)}")
print(f"Piastri clean laps:    {len(pia_laps)}")
print()
print("--- Overall Lap Time ---")
print(f"VER median: {ver_laps['LapTimeSec'].median():.3f}s")
print(f"PIA median: {pia_laps['LapTimeSec'].median():.3f}s")
gap = ver_laps['LapTimeSec'].median() - pia_laps['LapTimeSec'].median()
print(f"Gap (VER - PIA): {gap:+.3f}s")
print()
print("--- Sector 1 (contains Turn 1 from your IB IA) ---")
s1_gap = ver_laps['Sector1Sec'].median() - pia_laps['Sector1Sec'].median()
print(f"VER median S1: {ver_laps['Sector1Sec'].median():.3f}s")
print(f"PIA median S1: {pia_laps['Sector1Sec'].median():.3f}s")
print(f"S1 Gap: {s1_gap:+.3f}s")
print()
print("Next: 02_linear_regression.ipynb — predict lap time from tire age + race progress.")

---
## What We Used From Your Course

| Course concept | Where we used it | Why it mattered |
|---|---|---|
| **CRISP-DM** | Overall structure of the notebook | Keeps the project organized around a business question |
| **EDA** | Steps 7–11 | Revealed patterns stats alone wouldn't show |
| **Data Quality** | Step 5 | Missing values + inaccurate laps would corrupt our analysis |
| **Data Transformation** | Step 6 | timedelta → seconds so we can do math and feed into ML models later |
| **Anscombe's lesson** | Charted before summarizing | The line chart shows pit stop spikes that `.describe()` hides |

---
**Next notebook:** `02_linear_regression.ipynb` — predict lap time from tire age and lap number.